In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [2]:
# Uncomment and run this cell if you're on Colab or Kaggle
# !git clone https://github.com/nlp-with-transformers/notebooks.git
# %cd notebooks
# from install import *
# install_requirements()

In [3]:
#hide
from utils import *
setup_chapter()

Using transformers v4.57.1
Using datasets v5.0.0


# Multilingual Named Entity Recognition

## The Dataset

In [4]:
#id jeff-dean-ner
#caption An example of a sequence annotated with named entities
#hide_input
import pandas as pd
toks = "Jeff Dean is a computer scientist at Google in California".split()
lbls = ["B-PER", "I-PER", "O", "O", "O", "O", "O", "B-ORG", "O", "B-LOC"]
df = pd.DataFrame(data=[toks, lbls], index=['Tokens', 'Tags'])
df

,0,1,2,3,4,5,6,7,8,9
Tokens,Jeff,Dean,is,a,computer,scientist,at,Google,in,California
Tags,B-PER,I-PER,O,O,O,O,O,B-ORG,O,B-LOC


In [5]:
from datasets import get_dataset_config_names # Importa la función get_dataset_config_names()
# su misión-> Preguntar al Hub qué configuraciones (configs) tiene un dataset, sin descargarlo.

xtreme_subsets = get_dataset_config_names("xtreme")# "¿Qué configuraciones tiene el dataset xtreme?"
print(f"XTREME has {len(xtreme_subsets)} configurations")

XTREME has 183 configurations


In [6]:
panx_subsets = [s for s in xtreme_subsets if s.startswith("PAN")]
# for s in xtreme_subsets -> Recorre las distintas configuraciones
# Sólo selecciona las que empiezan por "PAM"
# s del principio -> Incluye la configuración seleccionada en la lista

panx_subsets[:3] # -> Muestra los 3 primeros elementos

['PAN-X.af', 'PAN-X.ar', 'PAN-X.bg']

Las configuraciones que empiezan por PAN-X corresponden precisamente al conjunto de datos de NER en distintos idiomas.

In [7]:
# hide_output
from datasets import load_dataset

load_dataset("xtreme", name="PAN-X.de") # Dentro de XTREME quiero la configuración PAN-X.de.

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 10000
    })
})

Objetivo de la siguiente celda

Construir un diccionario llamado panx_ch que contenga los datasets de varios idiomas, pero reduciendo el número de ejemplos de cada uno para que reflejen aproximadamente la distribución de idiomas hablados en Suiza.

Alemán  -> muchos ejemplos

Francés -> bastantes

Italiano -> menos

Inglés -> muy pocos


In [8]:
# hide_output
from collections import defaultdict # defaultdict es un diccionario "inteligente".
from datasets import DatasetDict # estructura train, validation, test

langs = ["de", "fr", "it", "en"] # idiomas que se van a usar
fracs = [0.629, 0.229, 0.084, 0.059] # proporciones de uso
# Return a DatasetDict if a key doesn't exist
panx_ch = defaultdict(DatasetDict) # por ejemplo si accedo a panx_ch[de] y todavía no existe -> crea un DatasetDict vacío
# así puedo escribir más adelante panx_ch[de]["train"] sin inicializar el diccionario antes

for lang, frac in zip(langs, fracs):
# Hacemos parejas ("de", 0.629)...
    # Load monolingual corpus
    ds = load_dataset("xtreme", name=f"PAN-X.{lang}") # esto sería el ful_ds -> abajo lo reducimos en panx_ch: Versión suiza
    # Shuffle and downsample each split according to spoken proportion
    for split in ds: #ds es un DatasetDict -> Itera en train, validation y test
        panx_ch[lang][split] = (
            ds[split]
            .shuffle(seed=0)#baraja los ejemplos ->tomo muestra aleatoria
            .select(range(int(frac * ds[split].num_rows))))
            #num_rows-> devuelve número de filas ej 20000
            #frac*ds[split].num_rows) -> toma sólo la fracción corresposnidente de esas filas ej 0.229 × 20000= 4580
            #int -> convierte el resultado en número entero -> 4580
            #.select(range(4580))-> se queda únicamente con esos 4.580 eje,mplos del dataset ya barajados
        

```text
panx_full
      │
      ▼
shuffle
      │
      ▼
select
      │
      ▼
panx_ch
```

In [9]:
# El objetivo es mostrar de forma bonita cuántos ejemplos de entrenamiento 
# hay en cada idioma después del muestreo que hicieron antes.

import pandas as pd

pd.DataFrame({lang: [panx_ch[lang]["train"].num_rows] for lang in langs},
             index=["Number of training examples"])


# Estructura es

###{clave: valor for ...}


# Los números entre corchetes porque en Pandas cada columna necesita una lista de valores
# Index -> cuando hay una sola fila: el nombre de esa fila

,de,fr,it,en
Number of training examples,12580,4580,1680,1180


Esta celda sirve para inspeccionar cómo es un ejemplo individual del dataset. Hasta ahora solo habíamos visto cuántos ejemplos había; ahora el libro quiere que veas la estructura de uno de ellos.

In [10]:
element = panx_ch["de"]["train"][0] # seleccionamos el primer ejemplo de train correpondiente al alemán
# este ejmeplo ese un diccionario

for key, value in element.items(): # .items devuelve los elementos del diccionario: claves y valores
    print(f"{key}: {value}")

tokens: ['2.000', 'Einwohnern', 'an', 'der', 'Danziger', 'Bucht', 'in', 'der',
'polnischen', 'Woiwodschaft', 'Pommern', '.']
ner_tags: [0, 0, 0, 0, 5, 6, 0, 0, 5, 5, 6, 0]
langs: ['de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de']


In [11]:
# ".feature" -> contiene la descripción del dataset, es decir, su esquema (schema)
# el esquema dice el tipo de dato de los items: string, integer...
for key, value in panx_ch["de"]["train"].features.items():
    print(f"{key}: {value}")

tokens: List(Value('string'))
ner_tags: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG',
'B-LOC', 'I-LOC']))
langs: List(Value('string'))


Detalle del esquema:

tokens: List(Value('string')) -> Lista de valores string

ner_tags: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG',
'B-LOC', 'I-LOC'])) -> Lista de ClassLabelm que es un tipo de dato

langs: List(Value('string'))-> -> Lista de valores string

La idea clave

Hay tres niveles de complejidad en los tipos de 🤗 Datasets:

- Value("string") → un valor simple (texto, entero, float...).
- List(...) o Sequence(...) → una colección de esos valores.
- ClassLabel(...) → un tipo especial que representa clases categóricas y mantiene automáticamente el mapeo entre enteros y nombres.


In [12]:
# seleccionamos la ner_tags festure del entrenamiento de "de" (alemán)
tags = panx_ch["de"]["train"].features["ner_tags"].feature
print(tags)

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])


In [13]:
# Para crear una nueva columna con los nombres de las etiquetas, en lugar de sus números
# hide_output
def create_tag_names(batch): # Esta función recibirá un ejemplo del dataset.
    return {"ner_tags_str": [tags.int2str(idx) for idx in batch["ner_tags"]]}
    # tags en la celda anterior ya conoce el mapeo:
    # por lo tanto tags.int2str(5)= B-LOC
    # .int2str(); convierte un entero(int) en nombre de la etiqueta (string).
    # Para hacer eso necesita tener un mapeo previo -> tags en celda anterior

panx_de = panx_ch["de"].map(create_tag_names) # ejecuta la función ceate_tag_names en cada ejemplo

Ahora tenemos:
- panx_ch -> Dataset original
- panx_de -> Dataset con una columna adicional

In [14]:
# hide_output
# Mostramos el primer ejemplo de train
    
de_example = panx_de["train"][0]
pd.DataFrame([de_example["tokens"], de_example["ner_tags_str"]],
['Tokens', 'Tags'])
# lista de listas -> conjunto de filas y luego establezco un alista con el nombre de las columnas

,0,1,2,3,4,5,6,7,8,9,10,11
Tokens,2.000,Einwohnern,an,der,Danziger,Bucht,in,der,polnischen,Woiwodschaft,Pommern,.
Tags,O,O,O,O,B-LOC,I-LOC,O,O,B-LOC,B-LOC,I-LOC,O


In [15]:
# si lo quisiera en columnas -> corchetes y esta estructura

pd.DataFrame({
    "Token": de_example["tokens"],
    "Tag": de_example["ner_tags_str"]
})

,Token,Tag
0,2.000,O
1,Einwohnern,O
2,an,O
3,der,O
4,Danziger,B-LOC
5,Bucht,I-LOC
6,in,O
7,der,O
8,polnischen,B-LOC
9,Woiwodschaft,B-LOC


In [16]:
# Calculamos la frecuencia de cada tipo de entidad (PER, LOC, ORG...)
# en cada split del dataset (train, validation y test)

from collections import Counter

# Counter es un diccionario especializado para contar elementos
# defaultdict(Counter) crea automáticamente un Counter vacío

split2freqs = defaultdict(Counter) # crea un diccionario inteligente que va sumando frecuencias
for split, dataset in panx_de.items():
    # panx_de ->DatasetDict con estructura separada en Train, Validation y Test
    # para cada split, su datataset
    for row in dataset["ner_tags_str"]:
        # para cada fila de la columna ["ner_tags_str"]
        for tag in row:
            # Recorremos cada etiqueta de la frase
            # Solo contamos las etiquetas B- porque representan
            if tag.startswith("B"): # si la tag comienza B
                tag_type = tag.split("-")[1] # hace split por "-" y pilla el segundo elemento  Separamos "B-LOC" -> ["B", "LOC"] Tomamos LOC
                split2freqs[split][tag_type] += 1 # añade la frecuencia al diccionario de frecuencias
                #incrementando el contador de ese tipo de entidad para el split correspondiente
pd.DataFrame.from_dict(split2freqs, orient="index") # las claves del diccionario (los splits) serán los índices (filas)

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071


Antes usé
- pd.DataFrame(...) -> Puedo meterle lista, diccionario o array_numpy

Ahora

- pd.DataFrame.from_dict(...) -> Método especializado cuando ya tengo un diccionario

d = {
    "A": [1,2,3],
    "B": [4,5,6]
}

pd.DataFrame.from_dict(d)

crea las filas 0 , 1 y 2 para las columnas A y B

In [17]:
# Tb podría haber hecho

pd.DataFrame(split2freqs).T # Transpongo el DF porque por defecto Pandas interpreta las claves como columnas

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071


## Multilingual Transformers

## A Closer Look at Tokenization

In [18]:
# hide_output
from transformers import AutoTokenizer

# cargamos los tokenizadores de los modelos BERT y XLM-R
bert_model_name = "bert-base-cased"
xlmr_model_name = "xlm-roberta-base"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
xlmr_tokenizer = AutoTokenizer.from_pretrained(xlmr_model_name)

In [19]:
# Para el mismo texto aplicamos el toenizador de
    #BERT -> WordPiece
    #XLM-R ->SentencePiece
text = "Jack Sparrow loves New York!"
bert_tokens = bert_tokenizer(text).tokens()
xlmr_tokens = xlmr_tokenizer(text).tokens()

In [20]:
#hide_input
# Mostramos las diferencias en un dataframe

df = pd.DataFrame([bert_tokens, xlmr_tokens], index=["BERT", "XLM-R"])
df

,0,1,2,3,4,5,6,7,8,9
BERT,[CLS],Jack,Spa,##rrow,loves,New,York,!,[SEP],NaN
XLM-R,<s>,▁Jack,▁Spar,row,▁love,s,▁New,▁York,!,</s>


### The Tokenizer Pipeline

<img alt="Tokenizer pipeline" caption="The steps in the tokenization pipeline" src="images/chapter04_tokenizer-pipeline.png" id="toknizer-pipeline"/>

### The SentencePiece Tokenizer

In [21]:
"".join(xlmr_tokens).replace(u"\u2581", " ")
# "".join(xlmr_tokens) -> junta todos los tokens con un vacío (sin separación)
# en SentencePiece los espacios -> "_" que se representa como U + 2581 (unicode))
# la primera u en u"\u2581" ->que lo siguiente es unicode
# \ -> Aquí empieza el código unicode
# Lo reemplazamos por un espacio
# Tb podría haber hecho .replace("▁", " ")

'<s> Jack Sparrow loves New York!</s>'

## Transformers for Named Entity Recognition

<img alt="Architecture of a transformer encoder for classification." caption="Fine-tuning an encoder-based transformer for sequence classification" src="images/chapter04_clf-architecture.png" id="clf-arch"/>

<img alt="Architecture of a transformer encoder for named entity recognition. The wide linear layer shows that the same linear layer is applied to all hidden states." caption="Fine-tuning an encoder-based transformer for named entity recognition" src="images/chapter04_ner-architecture.png" id="ner-arch"/>

## The Anatomy of the Transformers Model Class

### Bodies and Heads

<img alt="bert-body-head" caption="The `BertModel` class only contains the body of the model, while the `BertFor&lt;Task&gt;` classes combine the body with a dedicated head for a given task" src="images/chapter04_bert-body-head.png" id="bert-body-head"/>

### Creating a Custom Model for Token Classification

El autor quiere construir este modelo:

```text
                 XLM-RoBERTaForTokenClassification

                    ┌──────────────────────────┐
input_ids ─────────►│      XLM-R Body          │
                    └──────────────────────────┘
                               │
                    last_hidden_state
                               │
                               ▼
                    ┌──────────────────────────┐
                    │ Dropout + Linear (Head)  │
                    └──────────────────────────┘
                               │
                             logits
```


In [22]:
import torch.nn as nn # Importa los módulos de redes neuronales de PyTorch: Dropout, Linear, CrossEntropyLoss, etc.
from transformers import XLMRobertaConfig # Carga la clase de configuración de XLM-R.

from transformers.modeling_outputs import TokenClassifierOutput # Es el formato estándar de salida para modelos de clasificación de tokens.
from transformers.models.roberta.modeling_roberta import RobertaModel # importamos el modelo que hará de "body"
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel # importamos clase base de la que hereda el modelo

# Aquí usan las clases internas de RoBERTa porque XLM-R comparte arquitectura base con RoBERTa, 
# aunque esté entrenado de forma multilingüe.

class XLMRobertaForTokenClassification(RobertaPreTrainedModel): 
    # Hereda de RobertaPreTrainedModel, lo que le da funcionalidades de Hugging Face:
        # from_pretrained()
        # save_pretrained()
        # init_weights()
        # manejo de config. Simplemente se construye la arquitectura.
    
    config_class = XLMRobertaConfig # Esta clase usa configuraciones de XLM-RoBERTa. 
    # Es un metadato que usa HF internamente: 
    # Si alguna vez necesito crear una configuración para este modelo, la clase adecuada es XLMRobertaConfig.
        # la arquitectura de XLMR es la misma sólo cambia:
            # el vocabulario: SentencePiece en lugar de WordPiece
            # Los pesos entrenados
            # algunos parámetros de configuración

    
    def __init__(self, config):
        
        # Todo lo que aparece dentro del __init__ SE EJECUTA UNA ÚNICA VEZ, cuando construyes el modelo.
        # Aquí no se procesan frases.
        # El config que llega al __init__ es un objeto XLMRobertaConfig 
        # porque Hugging Face utiliza config_class para construirlo (o para comprobar que es del tipo correcto).
        
        
        super().__init__(config) # incializa clase padre(RobertaPreTrainedModel
        self.num_labels = config.num_labels # Guarda cuántas etiquetas NER hay (7).-> O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC
        
        # Éste es el body. Cargamos el modelo del body -> Heredamos de RobertaPreTrainedModel
        self.roberta = RobertaModel(config, add_pooling_layer=False) # porque en NER no necesitamos un vector global de frase. Necesitamos una salida por token.
        
        # Ésta es la head de clasificación
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels) # Para cada token pasa de hidden_size a num_label (7)
        # Load and initialize weights
        self.init_weights() # Inicializa los pesos nuevos, especialmente los de la cabeza clasificadora.

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, 
                labels=None, **kwargs):
        # el forward SE EJECUTA CADA VEZ QUE PASA UN BATCH
        # Use model body to get encoder representations
        
        kwargs.pop("num_items_in_batch", None) # me dio problemas num_items_in_batch -> Lo quito
        outputs = self.roberta(input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids, **kwargs)
        # -> Pasa los input_ids por el body -> shape (batch_size, sequence_length, hidden_size)
        # outputs no es un tensor, es un objeto similar a una tupla o diccionario que varias salidas
        # del modelo
            # [0] last_hidden_state -> Nos interesa ÉSTE
            # [1] pooler_output (si existe)
            # [2] hidden_states
            # [3] attentions


        
        # Apply classifier to encoder representation. Aquí empieza la cabeza
        sequence_output = self.dropout(outputs[0])
        logits = self.classifier(sequence_output)
        # Calculate losses
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss() # creamos función de pérdida
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            # los logits tiene shape (frases, tokens, clases por tokens) x ejemplo (32,128,7)
            # Pero CrossEntropy no acepta tensores 3D sino -> (N, num_classes)
            # logits.view(-1, self.num_labels) pasa los logits de (32, 128, 7) a (4096, 7).
            # logits.views(-1,7) da la forma a la segunda dimensión y con el "-1" se le pìde que calcule la 1a dim (32*128)
            # las etiquetas tienen shape (batch_size, sequence_lenght) x ejmeplo (32,128)
            # labels.view(-1)-> Las etiquetas pasan de (32,128) a (4096,)
        # Return model output object
        return TokenClassifierOutput(loss=loss, logits=logits, 
                                     hidden_states=outputs.hidden_states, 
                                     attentions=outputs.attentions)
        # empaqueta todos los resultados en un objeto estándar de Hugging Face.

CrossEntropyLoss no espera dos tensores con la misma forma. Espera precisamente estas dos formas:

input: (N, C)

target: (N)

donde:

N = número de ejemplos.

C = número de clases.

Construcción del modelo

↓

`__init__()`

(se ejecuta una vez)

---

Entrenamiento

↓

`forward()`

↓

`forward()`

↓

`forward()`

↓

`forward()`

### Loading a Custom Model

In [23]:
# Recordemos que 
# tags = panx_ch["de"]["train"].features["ner_tags"].feature
# Necesitamos de 2 mapeos:

    # 1) de índice a etiqueta -> Para ver resultados

index2tag = {idx: tag for idx, tag in enumerate(tags.names)}
    
    # 2) de etiqueta a índice para el entrenamiento

tag2index = {tag: idx for idx, tag in enumerate(tags.names)}

In [24]:
# hide_output
from transformers import AutoConfig

# Autoconfig -> Su trabajo es cargar la configuración del modelo

xlmr_config = AutoConfig.from_pretrained(xlmr_model_name, 
                                         num_labels=tags.num_classes,
                                         id2label=index2tag, label2id=tag2index)
# HF descarga la config de de ese modelo("xlm-roberta-base" que es un fichero JSON con Hidden_sizem num_hidden_layers...
# sobreescribimos algunos parámetros
    # num_labels=tags.num_classes: si no, no sabría cuantas clases tiene mi problema
    
    # añado los mapeos de antes para pasar de índice a etiqueta y de etiqueta a índice
    # Ahora el propio modelo conocerá esas correspondencias
        # model.config.id2label[5] ->"B-LOC"
        # model.config.label2id["B-ORG"] -> 3

In [25]:
# hide_output
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # elegimos dispositivo
xlmr_model = (XLMRobertaForTokenClassification
              .from_pretrained(xlmr_model_name, config=xlmr_config)
              .to(device))
# Antes definimos class XLMRobertaForTokenClassification(...):
# Ahora HF hace  algo como
    # config = xlmr_config
    # model = XLMRobertaForTokenClassification(config)

# es decir ejecuta __init__() + construye body + head + descarga pesos de xlm-roberta-base y los mete dentro del body
# Usa la config que personalizamos antes
# .to(device) -> lo manda todo a la GPU

In [26]:
# hide_output
# comprobamos que hemos inicializado el tokenizador correctamente
input_ids = xlmr_tokenizer.encode(text, return_tensors="pt")
pd.DataFrame([xlmr_tokens, input_ids[0].numpy()], index=["Tokens", "Input IDs"])

,0,1,2,3,4,5,6,7,8,9
Tokens,<s>,▁Jack,▁Spar,row,▁love,s,▁New,▁York,!,</s>
Input IDs,0,21763,37456,15555,5161,7,2356,5753,38,2


In [27]:
# Pasamos los inputs al modelo y extraemos la predicción para tomar con el argmax la clase más probable por token
outputs = xlmr_model(input_ids.to(device)).logits 
# input_ids.to(device) -> Enviamos las entradas a la GPU porque el modelo está en la GPU. antes: xlmr_model.to(device)
# PyTorch llama directamente a forward()
# .logits -> Antes:
    #return TokenClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions)
    # Por lo tanto el obejo TokenClassifierOutput contiene: loss, logits, hidden_states, attentions
    # Con .logist tomamos sólo los logits
    # forma de los logits, si la frase tiene 9 tokens -> (1,9,7) (batch_size, num_de_tokens, num_etiquetas_NER)




# Elegimos la clase más probable
predictions = torch.argmax(outputs, dim=-1)
print(f"Number of tokens in sequence: {len(xlmr_tokens)}")
print(f"Shape of outputs: {outputs.shape}")
print(f"Shape of predictions: {predictions.shape}")

Number of tokens in sequence: 10
Shape of outputs: torch.Size([1, 10, 7])
Shape of predictions: torch.Size([1, 10])


In [28]:
# Ahora queremos predecir las predicciones en nombres de etiquetas

preds = [tags.names[p] for p in predictions[0].cpu().numpy()] 
# predictions tiene shape (batch_size, sequence_length). Como sólo tenemos una frase -> [0]
# para convertir a numpy hya que pasar a la cpu desde la gpu con .cpu()
pd.DataFrame([xlmr_tokens, preds], index=["Tokens", "Tags"])

,0,1,2,3,4,5,6,7,8,9
Tokens,<s>,▁Jack,▁Spar,row,▁love,s,▁New,▁York,!,</s>
Tags,B-LOC,B-LOC,B-LOC,B-LOC,B-LOC,I-PER,I-PER,B-LOC,B-LOC,B-LOC


Podría haber hecho esto

preds = [
    model.config.id2label[p]
    for p in predictions[0].cpu().numpy()
]

Los resultados son malos porque todavía no hemos entrenado la cabeza de clasificación

Pero antes de eso vamos a envolver los pasos precedentes en una helper function para usarla después

In [29]:
def tag_text(text, tags, model, tokenizer):
    # Get tokens with special characters
    tokens = tokenizer(text).tokens()
    # Encode the sequence into IDs
    input_ids = xlmr_tokenizer(text, return_tensors="pt").input_ids.to(device)
    # Get predictions as distribution over 7 possible classes
    outputs = model(input_ids)[0]
    # Take argmax to get most likely class per token
    predictions = torch.argmax(outputs, dim=2)
    # Convert to DataFrame
    preds = [tags.names[p] for p in predictions[0].cpu().numpy()]
    return pd.DataFrame([tokens, preds], index=["Tokens", "Tags"])
    

## Tokenizing Texts for NER

Hasta ahora habíamos tokenizado y pasado al modelo una única frase, ahora tenemos que pasar el dataset completo para hacer fine-tuning

In [30]:
# Recordemos que de_example = panx_de["train"][0] -> la primera frase
words, labels = de_example["tokens"], de_example["ner_tags"]

In [31]:
tokenized_input = xlmr_tokenizer(de_example["tokens"], is_split_into_words=True)
# is_split_into_words=True -> Si no lo pusiéramos -> tokenizador interpretaría que le hosm paasdo un batch de frases
# Le decimos al tokenizador que no está dividido en frases sino en palabras
# Podría haber escrito xlmr_tokenizer(word, is_split_into_words=True)

tokens = xlmr_tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
# esto lo hace para pasar los tokens de ids (numericos) a tokens textuales para poder verlos
# tokenized_input contiene -> {"input_ids": [...],"words_id":[...], "attention_mask": [...]}



In [32]:
#hide_output
pd.DataFrame([tokens], index=["Tokens"])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>


In [33]:
# hide_output
word_ids = tokenized_input.word_ids() # Devuelve, para cada token, de qué palabra original proviene.
pd.DataFrame([tokens, word_ids], index=["Tokens", "Word IDs"])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Word IDs,NaN,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,NaN


Esta celda alinea las etiquetas originales de las palabras con los subtokens que ha creado XLM-R.

La idea es:

```text
Palabra original  → Etiqueta original
Subtokens         → Etiqueta solo en el primer subtoken
Resto             → -100 / IGN
```

In [34]:
#hide_output
previous_word_idx = None
label_ids = []

for word_idx in word_ids:
    if word_idx is None or word_idx == previous_word_idx:
        label_ids.append(-100)
    elif word_idx != previous_word_idx: #!= -> No es igual
        label_ids.append(labels[word_idx])
    previous_word_idx = word_idx # Actualiza la memoria para saber en la siguiente vuelta si seguimos en la misma palabra o hemos cambiado.
    
labels = [index2tag[l] if l != -100 else "IGN" for l in label_ids] #ojo l es una ele no un 1 (uno) 
# habíamos definido previamente index2tag = {idx: tag for idx, tag in enumerate(tags.names)}

index = ["Tokens", "Word IDs", "Label IDs", "Labels"]

pd.DataFrame([tokens, word_ids, label_ids, labels], index=index)

# recordemos: 
# O -> Outside, etiqueta real del Dataset -> "Esta palabra no pertenece a ninguna entidad nombrada."
# IGN -> "Ignorar". No es una etiqueta del dataset, equivale a -100 -> "No tengas en cuenta este token al calcular la pérdida."

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Word IDs,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None
Label IDs,-100,0,0,-100,0,0,5,-100,-100,6,...,5,-100,-100,-100,6,-100,-100,0,-100,-100
Labels,IGN,O,O,IGN,O,O,B-LOC,IGN,IGN,I-LOC,...,B-LOC,IGN,IGN,IGN,I-LOC,IGN,IGN,O,IGN,IGN


In [35]:
# Función que realiza todo lo anterior y lo escala a todo el dataset

def tokenize_and_align_labels(examples): # n recibe una frase sino un batch de ejemplos que son palabras separadas
    tokenized_inputs = xlmr_tokenizer(examples["tokens"], truncation=True, 
                                      is_split_into_words=True) # pasamos palabras ya separadas por cada ejemplo
    labels = []
    for idx, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=idx)
        previous_word_idx = None # iniclaizamos variable
        label_ids = [] # inicializamos variable
        for word_idx in word_ids:
            if word_idx is None or word_idx == previous_word_idx:
                label_ids.append(-100) # añade el -100 a label_ids -> ignorar
            else:
                label_ids.append(label[word_idx]) # añade la label_ids correspondiente
            previous_word_idx = word_idx 
        labels.append(label_ids) # "Saca" el resultado de la frase actual y lo guarda fuera del bucle interno, para empezar a procesar la siguiente frase.
    tokenized_inputs["labels"] = labels # añadir una nueva columna al objeto que ha devuelto el tokenizer.

    # antes: tokenized_inputs ->{"input_ids": [...],"attention_mask": [...]}
    # ahora: tokenized_inputs ->{"input_ids": [...],"attention_mask": [...], "labels": [[-100, 1, 2, 0, -100],[-100, 0, 3, -100]]}
    
    return tokenized_inputs
    

In [36]:
# Función wrapper que aplica todo lo de antes a todo el dataset

def encode_panx_dataset(corpus): 
    # corpus será un dataset de Hugging Face o incluso un DatasetDict con train, validation y test
    return corpus.map(tokenize_and_align_labels, batched=True, 
                      remove_columns=['langs', 'ner_tags', 'tokens'])
    #.map ejecuta la función tokenize_and_align_labels para todo el corpus
    # batched=True la función recibe varios ejemplos cada vez, no uno solo.
    # remove_columns -> antes de .map cada ejemplo tiene tokens, langs, ner_tags
    # despuñes de tokenize_and_align_labels, el resultado es:{
    #"input_ids": [...],"attention_mask": [...],"labels": [...]}
    # Ya no necesitamos las columnas originales -> Dataset más limpio

In [37]:
# hide_output
# Lo aplicamos a panx_ch["de"]
panx_de_encoded = encode_panx_dataset(panx_ch["de"])

Veámosla paso a paso.

## 1. `panx_ch["de"]`

Selecciona la parte del dataset correspondiente al alemán.

Es algo como:

```text
panx_ch
│
├── de
├── fr
├── it
├── en
└── ...
```

Y `panx_ch["de"]` contiene a su vez:

```text
train
validation
test
```

Es decir, es un `DatasetDict`:

```python
{
    "train": ...,
    "validation": ...,
    "test": ...
}
```

## 2. Se llama a `encode_panx_dataset(panx_ch["de"])`

Dentro de esa función ocurre:

```python
corpus.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=[...]
)
```

Como `corpus` es un `DatasetDict`, `map()` se aplica automáticamente a cada split:

```text
train
    ↓
tokenize_and_align_labels()

validation
    ↓
tokenize_and_align_labels()

test
    ↓
tokenize_and_align_labels()
```

No tienes que hacer un `for` tú mismo.

## 3. El resultado

`panx_de_encoded` sigue siendo un `DatasetDict`, pero transformado.

Antes:

```text
train
    tokens
    langs
    ner_tags

validation
    tokens
    langs
    ner_tags

test
    tokens
    langs
    ner_tags
```

Después:

```text
train
    input_ids
    attention_mask
    labels

validation
    input_ids
    attention_mask
    labels

test
    input_ids
    attention_mask
    labels
```

## Performance Measures

¿Qué es seqeval?

Es una librería especializada para evaluar tareas de Named Entity Recognition (NER) y otras tareas de etiquetado secuencial.

La diferencia con sklearn.metrics.classification_report es importante:

scikit-learn evalúa etiqueta por etiqueta (token a token).
seqeval evalúa entidades completas, que es lo que realmente interesa en NER.

Por ejemplo, si la entidad correcta es:

B-PER I-PER

y el modelo predice:

B-PER O

seqeval considera que la entidad está mal, aunque haya acertado uno de los tokens. Esa es la forma estándar de evaluar modelos NER.

In [38]:
# seqeval -> evalúa entidades completas
from seqeval.metrics import classification_report

y_true = [["O", "O", "O", "B-MISC", "I-MISC", "I-MISC", "O"],
          ["B-PER", "I-PER", "O"]]
y_pred = [["O", "O", "B-MISC", "I-MISC", "I-MISC", "I-MISC", "O"],
          ["B-PER", "I-PER", "O"]]
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

        MISC       0.00      0.00      0.00         1
         PER       1.00      1.00      1.00         1

   micro avg       0.50      0.50      0.50         2
   macro avg       0.50      0.50      0.50         2
weighted avg       0.50      0.50      0.50         2



In [39]:
# Esta función convierte la salida del modelo al formato que espera seqeval.

import numpy as np

def align_predictions(predictions, label_ids):
    # predictions.shape -> (batch_size, seq_len, num_labels)
    preds = np.argmax(predictions, axis=2) # Nos quedamos con la etiqueta más probable (3ª dimensión)
    # Ahora ya no tenemos probabilidades sino índices de etiquetas
    
    batch_size, seq_len = preds.shape # preds.shape (batch_size, seq_len). Obtenemos cada dimensión
    labels_list, preds_list = [], [] # creamos listas vacías donde almacenará el resultado final
    
    for batch_idx in range(batch_size): # Recorre cada ejemplo
        example_labels, example_preds = [], [] # Estas listas contendrán únicamente la frase actual.
        for seq_idx in range(seq_len):
            # Ignore label IDs = -100 
            if label_ids[batch_idx, seq_idx] != -100:
                example_labels.append(index2tag[label_ids[batch_idx][seq_idx]])
                example_preds.append(index2tag[preds[batch_idx][seq_idx]])
            # si IDs =100 -> vacío

        # se pasan a las listas principales
        labels_list.append(example_labels) 
        preds_list.append(example_preds)

    return preds_list, labels_list

## Fine-Tuning XLM-RoBERTa

nuestra pirmera estrategia es hace fine tuning del modelo base en el subset alemán de PAN-X y después evaluar su zero crosslingual performance en francés inglés e italiano.

Es decir entrenamos primero en aleman 

panx_de_encoded["train"]

y luego preguntas:

¿Lo aprendido en alemán se transfiere a francés, italiano e inglés sin haber entrenado con esos idiomas?

Eso es lo que llaman zero-shot cross-lingual transfer.

La idea general es:

- TrainingArguments = configuración del entrenamiento
- Trainer = objeto que ejecuta el entrenamiento
- trainer.train() = empieza realmente el fine-tuning

In [40]:
# hide_output

# Definimos primero el trainer para manejar el traing loop.
# Necesitamos definir primero los atributos de entrenamiento usando la clase TrainingArguments Class

from transformers import TrainingArguments

num_epochs = 3 # El modelo verá el dataset de entrenamiento alemán 3 veces completas.
batch_size = 24 # El modelo procesará 24 ejemplos por paso de entrenamiento.
logging_steps = len(panx_de_encoded["train"]) // batch_size # calcula aproximadamente cuántos batches hay en una época
model_name = f"{xlmr_model_name}-finetuned-panx-de" # ombre del directorio donde se guardará el modelo
training_args = TrainingArguments(
    output_dir=model_name, log_level="error", num_train_epochs=num_epochs, 
    per_device_train_batch_size=batch_size, 
    per_device_eval_batch_size=batch_size, eval_strategy="epoch", 
    save_steps=1e6, weight_decay=0.01, disable_tqdm=False, 
    logging_steps=logging_steps, push_to_hub=True)

# Argumentos principales
# output_dir=model_name -> Carpeta donde se guardan checkpoints, logs y modelo final.
# log_level="error" -> Reduce los mensajes de Hugging Face. Solo muestra errores importantes.
# per_device_train_batch_size=batch_size -> Batch size de entrenamiento por dispositivo
# per_device_eval_batch_size=batch_size -> Batch size de entrenamiento por dispositivo
# evaluation_strategy="epoch" -> Evalúa el modelo al final de cada época -> ojo: eval_strategy="epoch"
# save_steps=1e6 -> guarda un checkpoint cada 1.000.000 pasos
# weight_decay=0.01 -> Regularización L2-> Ayuda a que el modelo no sobreajuste demasiado al alemán
# disable_tqdm=False -> Permite var la barra de progreso
# logging_steps=logging_steps -> Cada cierto número de pasos imprimirá métricas / Logs
# push_to_hub=True -> Al final, o durante el entrenamiento según configuración, puede subir el modelo a Hugging Face Hub


In [41]:
# Para comprobar que me conecto a HF
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '67a649f497046c12fb81f911', 'name': 'srmjfba',
'fullname': 'Juan Francisco Barragán', 'email': 'srmjfba@gmail.com',
'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd':
1785542400, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/
production/uploads/no-auth/YVY2t-i2sr2IOjzV0ZDD-.png', 'orgs': [], 'auth':
{'type': 'access_token', 'accessToken': {'displayName': 'NLP with Transformers',
'role': 'write', 'createdAt': '2026-06-29T17:08:26.812Z'}}}


In [42]:
#hide_output
from huggingface_hub import notebook_login

notebook_login()

In [43]:
# Esta función es la que el Trainer llamará automáticamente cada vez que evalúe el modelo

from seqeval.metrics import f1_score # Importamos la métrica F1 específica para NER.

def compute_metrics(eval_pred):
    # eval_pred -> objeto que contiene las predicciones del modelo y las etiquestas reales del conjunto de validación
    y_pred, y_true = align_predictions(eval_pred.predictions, 
                                       eval_pred.label_ids)
    return {"f1": f1_score(y_true, y_pred)} # Aquí seqeval compara las entidades reales con las predichas y devuelve un único número

El Data Collator es una pieza del Trainer.

- Un Data Collator toma varios ejemplos individuales del dataset y los convierte en un batch listo para entrar en el modelo.
- Añade pads a los ejemplos con menos longitud que el que tiene más longitud del batch
- PyTorch no exige que todos los batches tengan la misma longitud. Solo exige que dentro de un mismo batch todas las secuencias tengan la misma longitud para poder formar un tensor.

In [44]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(xlmr_tokenizer)

In [45]:
def model_init():
    return (XLMRobertaForTokenClassification
            .from_pretrained(xlmr_model_name, config=xlmr_config)
            .to(device))

# Es como una fábrica de modelos.
# Cada vez que alguien hace -> model = model_init()

In [46]:
#hide
%env TOKENIZERS_PARALLELISM=false

# No sale en el libro


env: TOKENIZERS_PARALLELISM=false


Debo usarlo?
✅ Si ves advertencias sobre TOKENIZERS_PARALLELISM, puedes dejar esa línea.
✅ Si no aparece ninguna advertencia, normalmente puedes omitirla y todo funcionará igual.

In [47]:
# hide_output
from transformers import Trainer

trainer = Trainer(model_init=model_init, args=training_args, 
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  train_dataset=panx_de_encoded["train"],
                  eval_dataset=panx_de_encoded["validation"], 
                  tokenizer=xlmr_tokenizer)

# model_init= model_init y no = model_init() porque el trainer guarda esa función
# cuando empiece el entrenamiento internamente hará model= self.model_init()
# args=training_args -> aquí están todas las opciones del entrenamiento (learning rate, batch_size, epochs..)
# data_collator=data_collator- > hace que el modelo reciba tensores rectangulares
# compute_metrics=compute_metrics -> Llama a la función anterior.
# cuando termine una evaluación trainer.evaluate -> precciones y etiquetas reales
# entonces llamará a compute_metrics(eval_pred) y obtendrá f1
# train_dataset=panx_de_encoded["train"] -> conj de entrenamiento
# eval_dataset=panx_de_encoded["validation"] -> conjutno de validación
# tokenizer=xlmr_tokenizer -> Ya hemos tokenizado pero el trainer lo necesita para
    # algunas tareas de predcción
    # padding dinámcio en ciertos casos
    # facilitar la inferencia posterior


/tmp/ipykernel_557/1587757416.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model_init=model_init, args=training_args,


El Trainer hace internamente un montón de trabajo:

- Crea el modelo con model_init().
- Construye los DataLoader.
- Forma los batches usando el data_collator.
- Ejecuta el entrenamiento.
- Calcula la loss.
- Actualiza los pesos con el optimizador.
- Evalúa periódicamente con eval_dataset.
- Calcula el F1 usando compute_metrics.
- Guarda checkpoints si así lo indican training_args.

Ese es precisamente **el valor de Trainer: encapsula todo el bucle de entrenamiento que en PyTorch "puro" tendrías que escribir manualmente**. A partir de este punto del libro, ya no vas a ver tanto el detalle del entrenamiento, porque Trainer se encarga de toda esa lógica.

In [48]:
#hide_input
# ejecuta el entrenamiento y el push to hub 
trainer.train()


Epoch,Training Loss,Validation Loss,F1
1,0.259800,0.154560,0.822833
2,0.127100,0.142667,0.847020
3,0.081700,0.137885,0.863243


TrainOutput(global_step=1575, training_loss=0.15608755654758877, metrics={'train_runtime': 246.5039, 'train_samples_per_second': 153.101, 'train_steps_per_second': 6.389, 'total_flos': 862324400720376.0, 'train_loss': 0.15608755654758877, 'epoch': 3.0})

In [49]:
trainer.push_to_hub(commit_message="Training completed!") # sube todo lo necesario para reutilizar el modelo
    # config.json
    # model.safetensors
    # tokenizer.json
    # tokenizer_config.json
    # special_tokens_map.json
    # README.md

Upload 0 LFS files: 0it [00:00, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/srmjfba/xlm-roberta-base-finetuned-panx-de/commit/80c9256b20c7852d368941727920735b4d8db930', commit_message='Training completed!', commit_description='', oid='80c9256b20c7852d368941727920735b4d8db930', pr_url=None, repo_url=RepoUrl('https://huggingface.co/srmjfba/xlm-roberta-base-finetuned-panx-de', endpoint='https://huggingface.co', repo_type='model', repo_id='srmjfba/xlm-roberta-base-finetuned-panx-de'), pr_revision=None, pr_num=None)

In [50]:
# hide_input
# forma elegante de extraer el historial del entrenamiento que ha ido guardando el Trainer 
# y convertirlo en una tabla fácil de lee

df = pd.DataFrame(trainer.state.log_history)[['epoch','loss' ,'eval_loss', 'eval_f1']]
df = df.rename(columns={"epoch":"Epoch","loss": "Training Loss", "eval_loss": "Validation Loss", "eval_f1":"F1"})
df['Epoch'] = df["Epoch"].apply(lambda x: round(x))
df['Training Loss'] = df["Training Loss"].ffill()
df[['Validation Loss', 'F1']] = df[['Validation Loss', 'F1']].bfill().ffill()
df.drop_duplicates()

,Epoch,Training Loss,Validation Loss,F1
0,1,0.2598,0.154560,0.822833
2,2,0.1271,0.142667,0.847020
4,3,0.0817,0.137885,0.863243


In [51]:
# hide_output

# Usamos el modelo para inferencia
# Antes definimos-> def tag_text(text, tags, model, tokenizer):

text_de = "Jeff Dean ist ein Informatiker bei Google in Kalifornien"
tag_text(text_de, tags, trainer.model, xlmr_tokenizer)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
Tokens,<s>,▁Jeff,▁De,an,▁ist,▁ein,▁Informati,ker,▁bei,▁Google,▁in,▁Kaliforni,en,</s>
Tags,O,B-PER,I-PER,I-PER,O,O,O,O,O,B-ORG,O,B-LOC,I-LOC,O


## Error Analysis

Esta función sirve para hacer análisis de errores a nivel token.

Hasta ahora el modelo daba una métrica global:

F1 = 0.86

Pero eso no te dice dónde se equivoca.

Esta función intenta responder:

“¿En qué tokens concretos el modelo tiene más pérdida?”

Idea general

Para cada ejemplo del dataset, queremos obtener:

- token real
- etiqueta real
- etiqueta predicha
- loss del token

Así luego podemos ordenar por loss y ver los casos más problemáticos.

In [61]:
# función para ver los ejemplos con más errores. 
# pero ahora calcularemos el error por token la secuencia del ejemplo

from torch.nn.functional import cross_entropy # Import la f. d pérdida q se usa normalm en clasif: cross entropy.

def forward_pass_with_label(batch): # f. q recibirá un batch (varias frases juntas) del dataset
    # Convert dict of lists to list of dicts suitable for data collator
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
   
     # el batch viene así -> como un diccionario de listas
    
    # {"input_ids": [[...], [...], [...]],
    # "attention_mask": [[...], [...], [...]],
    # "labels": [[...], [...], [...]]}

    # Pero el data collator espera -> lista de diccionarios
    
    # [{"input_ids": [...], "attention_mask": [...], "labels": [...]},
    # {"input_ids": [...], "attention_mask": [...], "labels": [...]},
    # {"input_ids": [...], "attention_mask": [...], "labels": [...]}]

    # batch.values()-> devuelve los valores sin los nombres "input_ids", "attention_mask" y "labels"
    # zip -> recibe varios iterables y va agrupando el primer elemento de cada uno,
    # luego el segundo, luego el tercero..
    # * -> desempaqueta los valores de batch.values() para que zip reciba cada lista como un argumento independiente
    # ahora cada "t" -> 1 ejemplo completo
    # * -> convierte zip(batch.values()) en algo equivalente a zip(lista1, lista2, lista3)

    

    # zip(batch, t) -> itera sobre las claves batch, obteniendo las claves que quitamos antes y lo junta con los valores t de antes
        # ("input_ids", [...])
        # ("attention_mask", [...])
        # ("labels", [...])

    # dict -> Lo convierte en diccionario 
    
    
    
    # Pad inputs and labels and put all tensors on device
    batch = data_collator(features)
    # Pasa las features (lista de diccionarios) por el data_collator
    # data_collator -> hace lo siguiente:
        # 1) calcula la longitud máxima del batch 
        # 2) hace padding de cada ejemplo de ese batch hasta llegar a la longitud máxima
        # 3) Convierte tod en tensores de PyTorch
            # batch = {
            # "input_ids": tensor(...),
            # "attention_mask": tensor(...),
            # "labels": tensor(...) }
        # 4) Machaca la variable batch -> ahora es un diccionario de tensores con padding
       

    
    input_ids = batch["input_ids"].to(device) # extrae el tensor input_ids y lo envía a la GPU
    attention_mask = batch["attention_mask"].to(device) # extrae el tensor attention_mask y lo envía a la GPU
    labels = batch["labels"].to(device) # extrae el tensor attention_mask y lo envía a la GPU

    # Lo enviamos a la GPU porque el modelo está allí

    with torch.no_grad(): # Voy a usar el modelo sólo para predecir, no para entrenarlo (no hacemos backward)
        # Pass data through model  
        output = trainer.model(input_ids, attention_mask) # pasamos inputs_ids y attention_mask del bacth
        # output.logits -> objeto que contiene las puntuaciones
        # Logits.size: [batch_size, sequence_length, classes]
        # Predict class with largest logit value on classes axis
        predicted_label = torch.argmax(output.logits, axis=-1).cpu().numpy() # elegimos la clase con el output.logits más alto
        # Para convertirlo de tensor a NumPy primero hay que traerlo a la CPU

        
        
    # Calculate loss per token after flattening batch dimension with view
    loss = cross_entropy(output.logits.view(-1, 7), 
                         labels.view(-1), reduction="none")
   
    # Queremos calcular un error para cada token
    # Los logits tienen forma [batch, tokens, clases]
    # Pero cross_entropy espera las predicciones así -> [batch, clases]
    # output.logits.view(-1, 7) -> Mantiene la dimensión de las clases (7 clases) y calcula la 1ª dim automáticamnete (-1)
    # La 1ª dim = batch*tokens -> (batch*tokens, clases)
    # cross_entropy(...) -> compara para cada token -> logits predichos vs clase correcta 
    # Si el modelo dio mucha puntuación a la clase real -> la pérdida será baja.
    # Si el modelo dio poca puntuación a la clase real -> la pérdida será alta.
    # reduction = "none" -> Normalmente cross_entropy devolvería una sola media, Con esto -> Pérdida por token
    # La salida de loss tiene forma (token)



    # Unflatten batch dimension and convert to numpy array
    loss = loss.view(len(input_ids), -1).cpu().numpy()
    # input_ids tiene shape(batch, sequence_lenght)
    # len(input_ids) cuenta el número de elementos de la primera dimensión
    # loss.view(len(input_ids), -1) -> le doy como 1ª dim el tamaño dl batch y me recalcula la segunda

    return {"loss":loss, "predicted_label": predicted_label}

In [62]:
# hide_output
valid_set = panx_de_encoded["validation"] # seleccionamos el split de validación del dataset codificado
valid_set = valid_set.map(forward_pass_with_label, batched=True, batch_size=32) 
# batched= True -> la f(x) no recibe un ejemplo sino un lote de ejemplos
# le aplicamos la función anterior en lots de 32 frases
df = valid_set.to_pandas() # Convierte el Dataset de Hugging Face en un DataFrame de pandas.

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

**¿Qué hace map() con ese resultado?**

Lo añade como nuevas columnas al dataset.

Antes el dataset tenía algo parecido a:

- input_ids
- attention_mask
- labels

Después tendrá:

- input_ids
- attention_mask
- labels
- loss
- predicted_label

No reemplaza las columnas anteriores, simplemente añade estas dos.

### ¿Por qué calculamos la *loss* después del entrenamiento?

Durante el entrenamiento, el modelo ya calcula la *loss*, pero obtiene **una única pérdida media por batch**, que es la que utiliza para actualizar los pesos mediante *backpropagation*.

Una vez entrenado el modelo, recalculamos la *loss* sobre el conjunto de validación con `reduction="none"` para obtener **una pérdida por cada token**. Esto permite realizar un análisis detallado de errores:

- Identificar los tokens con mayor error.
- Detectar las frases más difíciles para el modelo.
- Analizar posibles errores de etiquetado o casos ambiguos.
- Evaluar el comportamiento del **modelo final**, no el de una etapa intermedia del entrenamiento.

En resumen:

- **Entrenamiento:** una *loss* media → optimizar los pesos del modelo.
- **Post-entrenamiento:** una *loss* por token → analizar y comprender los errores del modelo.

In [63]:
df.head()


,input_ids,attention_mask,labels,loss,predicted_label
0,"[0, 10699, 11, 15, 16104, 1388, 2]","[1, 1, 1, 1, 1, 1, 1]","[-100, 3, -100, 4, 4, 4, -100]","[0.0, 0.00923292, 0.0, 0.019512607, 0.01776603...","[4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ..."
1,"[0, 56530, 25216, 30121, 152385, 19229, 83982,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 0, -100, -100, -100, -100, 3, -100, -10...","[0.0, 0.00027688485, 0.0, 0.0, 0.0, 0.0, 1.915...","[6, 0, 0, 6, 0, 0, 1, 6, 2, 6, 2, 2, 2, 6, 2, ..."
2,"[0, 159093, 165, 38506, 122, 153080, 29088, 57...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[-100, 0, 0, 0, 0, 3, -100, -100, 0, -100, 0, ...","[0.0, 0.00013469743, 9.9654004e-05, 0.00011848...","[0, 0, 0, 0, 0, 3, 0, 4, 0, 0, 0, 0, 0, 0, 0, ..."
3,"[0, 16459, 242, 5106, 6, 198715, 5106, 242, 2]","[1, 1, 1, 1, 1, 1, 1, 1, 1]","[-100, 0, 0, 0, 5, -100, 0, 0, -100]","[0.0, 0.00015543684, 0.00013290952, 0.00016532...","[5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"[0, 11022, 2315, 7418, 1079, 8186, 57242, 97, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 0, 0, 0, 0, 0, 0, 0, -100, 0, 0, 0, 3, ...","[0.0, 0.000107282605, 0.00010144196, 0.0001097...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 4, ..."


In [ ]:
# hide_output
# Objetivo -> Hacer el DataFrame legible para que podamos analizar los errores.

index2tag[-100] = "IGN" # añade una etiqueta para -100
df["input_tokens"] = df["input_ids"].apply(
    lambda x: xlmr_tokenizer.convert_ids_to_tokens(x))

# df["input_tokens"] -> crea una nueva columna llamada "input_tokens"
# Esta columna sera el resultado de aplicar el convert_ids_to_tokens() del tokenizador
# Pasamos de [0, 8774, 20456, 2] a ["<s>", "John", "Smith", "</s>"]
# La función convert_ids_to_tokens() está diseñada para aceptar directamente una lista de IDs.


df["predicted_label"] = df["predicted_label"].apply(
    lambda x: [index2tag[i] for i in x])
# Modifica la columna "predicted_label" (ya existe)
# Esta columna será el resultado de aplicar a cada elemento de la lista el diccionario index2tag.
# x es cada lista e i cada predicción

df["labels"] = df["labels"].apply(
    lambda x: [index2tag[i] for i in x])
# hace lo mismo con la columna "labels"
df['loss'] = df.apply(
    lambda x: x['loss'][:len(x['input_ids'])], axis=1)
# x -> una fila completa del DataFrame

    # x =
    
    # input_ids          [0, 1234, 5678, 2]
    # loss               [0.02, 0.10, 1.80, 0.01, 0.00, 0.00]
    # labels             [...]
    # predicted_label    [...]
    # ...

# x["loss"] -> Obtiene la lista de pérdidas de esa fila. 

    #[0.02, 0.10, 1.80, 0.01, 0.00, 0.00]

# len(x["input_ids"]) -> Nº de elementos de input_ids -> en este ejemplo = 4

# x["loss"][:4] -> Toma los 4 primeros elementos de loss, depreciando los últimos 2 -> [0.02, 0.10, 1.80, 0.01
    # Porque los 2 últimos son pads 

df['predicted_label'] = df.apply(
    lambda x: x['predicted_label'][:len(x['input_ids'])], axis=1)
# hace los mismo con "predicted_label"

df.head(1)

In [ ]:
# hide_output
df_tokens = df.apply(pd.Series.explode)
df_tokens = df_tokens.query("labels != 'IGN'")
df_tokens["loss"] = df_tokens["loss"].astype(float).round(2)
df_tokens.head(7)

In [ ]:
(
    df_tokens.groupby("input_tokens")[["loss"]]
    .agg(["count", "mean", "sum"])
    .droplevel(level=0, axis=1)  # Get rid of multi-level columns
    .sort_values(by="sum", ascending=False)
    .reset_index()
    .round(2)
    .head(10)
    .T
)

In [ ]:
(
    df_tokens.groupby("labels")[["loss"]] 
    .agg(["count", "mean", "sum"])
    .droplevel(level=0, axis=1)
    .sort_values(by="mean", ascending=False)
    .reset_index()
    .round(2)
    .T
)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

def plot_confusion_matrix(y_preds, y_true, labels):
    cm = confusion_matrix(y_true, y_preds, normalize="true")
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
    plt.title("Normalized confusion matrix")
    plt.show()

In [ ]:
plot_confusion_matrix(df_tokens["labels"], df_tokens["predicted_label"],
                      tags.names)

In [ ]:
# hide_output
def get_samples(df):
    for _, row in df.iterrows():
        labels, preds, tokens, losses = [], [], [], []
        for i, mask in enumerate(row["attention_mask"]):
            if i not in {0, len(row["attention_mask"])}:
                labels.append(row["labels"][i])
                preds.append(row["predicted_label"][i])
                tokens.append(row["input_tokens"][i])
                losses.append(f"{row['loss'][i]:.2f}")
        df_tmp = pd.DataFrame({"tokens": tokens, "labels": labels, 
                               "preds": preds, "losses": losses}).T
        yield df_tmp

df["total_loss"] = df["loss"].apply(sum)
df_tmp = df.sort_values(by="total_loss", ascending=False).head(3)

for sample in get_samples(df_tmp):
    display(sample)

In [ ]:
# hide_output
df_tmp = df.loc[df["input_tokens"].apply(lambda x: u"\u2581(" in x)].head(2)
for sample in get_samples(df_tmp):
    display(sample)

## Cross-Lingual Transfer

In [ ]:
def get_f1_score(trainer, dataset):
    return trainer.predict(dataset).metrics["test_f1"]

In [ ]:
f1_scores = defaultdict(dict)
f1_scores["de"]["de"] = get_f1_score(trainer, panx_de_encoded["test"])
print(f"F1-score of [de] model on [de] dataset: {f1_scores['de']['de']:.3f}")

In [ ]:
text_fr = "Jeff Dean est informaticien chez Google en Californie"
tag_text(text_fr, tags, trainer.model, xlmr_tokenizer)

In [ ]:
def evaluate_lang_performance(lang, trainer):
    panx_ds = encode_panx_dataset(panx_ch[lang])
    return get_f1_score(trainer, panx_ds["test"])

In [ ]:
# hide_output
f1_scores["de"]["fr"] = evaluate_lang_performance("fr", trainer)
print(f"F1-score of [de] model on [fr] dataset: {f1_scores['de']['fr']:.3f}")

In [ ]:
# hide_input
print(f"F1-score of [de] model on [fr] dataset: {f1_scores['de']['fr']:.3f}")

In [ ]:
# hide_output
f1_scores["de"]["it"] = evaluate_lang_performance("it", trainer)
print(f"F1-score of [de] model on [it] dataset: {f1_scores['de']['it']:.3f}")

In [ ]:
# hide_input
print(f"F1-score of [de] model on [it] dataset: {f1_scores['de']['it']:.3f}")

In [ ]:
#hide_output
f1_scores["de"]["en"] = evaluate_lang_performance("en", trainer)
print(f"F1-score of [de] model on [en] dataset: {f1_scores['de']['en']:.3f}")

In [ ]:
#hide_input
print(f"F1-score of [de] model on [en] dataset: {f1_scores['de']['en']:.3f}")

### When Does Zero-Shot Transfer Make Sense?

In [ ]:
def train_on_subset(dataset, num_samples):
    train_ds = dataset["train"].shuffle(seed=42).select(range(num_samples))
    valid_ds = dataset["validation"]
    test_ds = dataset["test"]
    training_args.logging_steps = len(train_ds) // batch_size
    
    trainer = Trainer(model_init=model_init, args=training_args,
        data_collator=data_collator, compute_metrics=compute_metrics,
        train_dataset=train_ds, eval_dataset=valid_ds, tokenizer=xlmr_tokenizer)
    trainer.train()
    if training_args.push_to_hub:
        trainer.push_to_hub(commit_message="Training completed!")
    
    f1_score = get_f1_score(trainer, test_ds)
    return pd.DataFrame.from_dict(
        {"num_samples": [len(train_ds)], "f1_score": [f1_score]})

In [ ]:
# hide_output
panx_fr_encoded = encode_panx_dataset(panx_ch["fr"])

In [ ]:
# hide_output
training_args.push_to_hub = False
metrics_df = train_on_subset(panx_fr_encoded, 250)
metrics_df

In [ ]:
#hide_input
# Hack needed to exclude the progress bars in the above cell
metrics_df

In [ ]:
# hide_output
for num_samples in [500, 1000, 2000, 4000]:
    metrics_df = metrics_df.append(
        train_on_subset(panx_fr_encoded, num_samples), ignore_index=True)

In [ ]:
fig, ax = plt.subplots()
ax.axhline(f1_scores["de"]["fr"], ls="--", color="r")
metrics_df.set_index("num_samples").plot(ax=ax)
plt.legend(["Zero-shot from de", "Fine-tuned on fr"], loc="lower right")
plt.ylim((0, 1))
plt.xlabel("Number of Training Samples")
plt.ylabel("F1 Score")
plt.show()

### Fine-Tuning on Multiple Languages at Once

In [ ]:
from datasets import concatenate_datasets

def concatenate_splits(corpora):
    multi_corpus = DatasetDict()
    for split in corpora[0].keys():
        multi_corpus[split] = concatenate_datasets(
            [corpus[split] for corpus in corpora]).shuffle(seed=42)
    return multi_corpus

In [ ]:
panx_de_fr_encoded = concatenate_splits([panx_de_encoded, panx_fr_encoded])

In [ ]:
# hide_output
training_args.logging_steps = len(panx_de_fr_encoded["train"]) // batch_size
training_args.push_to_hub = True
training_args.output_dir = "xlm-roberta-base-finetuned-panx-de-fr"

trainer = Trainer(model_init=model_init, args=training_args,
    data_collator=data_collator, compute_metrics=compute_metrics,
    tokenizer=xlmr_tokenizer, train_dataset=panx_de_fr_encoded["train"],
    eval_dataset=panx_de_fr_encoded["validation"])

trainer.train()
trainer.push_to_hub(commit_message="Training completed!")

In [ ]:
#hide_output
for lang in langs:
    f1 = evaluate_lang_performance(lang, trainer)
    print(f"F1-score of [de-fr] model on [{lang}] dataset: {f1:.3f}")

In [ ]:
#hide_input
for lang in langs:
    f1 = evaluate_lang_performance(lang, trainer)
    print(f"F1-score of [de-fr] model on [{lang}] dataset: {f1:.3f}")

In [ ]:
# hide_output
corpora = [panx_de_encoded]

# Exclude German from iteration
for lang in langs[1:]:
    training_args.output_dir = f"xlm-roberta-base-finetuned-panx-{lang}"
    # Fine-tune on monolingual corpus
    ds_encoded = encode_panx_dataset(panx_ch[lang])
    metrics = train_on_subset(ds_encoded, ds_encoded["train"].num_rows)
    # Collect F1-scores in common dict
    f1_scores[lang][lang] = metrics["f1_score"][0]
    # Add monolingual corpus to list of corpora to concatenate
    corpora.append(ds_encoded)

In [ ]:
corpora_encoded = concatenate_splits(corpora)

In [ ]:
# hide_output
training_args.logging_steps = len(corpora_encoded["train"]) // batch_size
training_args.output_dir = "xlm-roberta-base-finetuned-panx-all"

trainer = Trainer(model_init=model_init, args=training_args,
    data_collator=data_collator, compute_metrics=compute_metrics,
    tokenizer=xlmr_tokenizer, train_dataset=corpora_encoded["train"],
    eval_dataset=corpora_encoded["validation"])

trainer.train()
trainer.push_to_hub(commit_message="Training completed!")

In [ ]:
# hide_output
for idx, lang in enumerate(langs):
    f1_scores["all"][lang] = get_f1_score(trainer, corpora[idx]["test"])

In [ ]:
scores_data = {"de": f1_scores["de"],
               "each": {lang: f1_scores[lang][lang] for lang in langs},
               "all": f1_scores["all"]}
f1_scores_df = pd.DataFrame(scores_data).T.round(4)
f1_scores_df.rename_axis(index="Fine-tune on", columns="Evaluated on",
                         inplace=True)
f1_scores_df

## Interacting with Model Widgets

<img alt="A Hub widget" caption="Example of a widget on the Hugging Face Hub" src="images/chapter04_ner-widget.png" id="ner-widget"/>  

## Conclusion